<a href="https://colab.research.google.com/github/SaiOhmSaiePaine/food_not_food_text_classifier/blob/main/Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Turning a model into a demo

In [1]:
try:
  import gradio as gr
  import transformers
except ModuleNotFoundError:
  !pip install -q transformers gradio
  import gradio as gr

# Creating a simple function to perform inference

In [2]:
import os
import torch
from transformers import pipeline
from typing import Dict


# 1. Define configuration variables OUTSIDE the function
HUGGING_FACE_MODEL_PATH = os.getenv(
    "MODEL_PATH",
    "SaiePaine/learn_hf_food_not_food_text_classifier-distilbert-base-uncased"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# 2. Initialize the pipeline ONCE outside the function (at application startup)
# It loads the model into memory exactly once.
classifier_pipeline = pipeline(
    task='text-classification',
    model=HUGGING_FACE_MODEL_PATH,
    device=DEVICE,
    top_k=None,
    batch_size=32
)


# 3. Create a function which takes text as input
def food_not_food_classifier(text: str) -> Dict[str, float]:
  """
  Take an input string of text and classifies it into food/not_food in the forms of a dictionary.
  """

  # Get outputs from pipeline (as a list of dicts)
  outputs = classifier_pipeline(text)[0]

  # Format output for Gradio (e.g. {"label_1": probability_1, "label_2": probability_2})
  output_dict = {}
  for item in outputs:
    output_dict[item['label']] = item["score"]

  return output_dict

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

In [3]:
# Test out the function
food_not_food_classifier("I don't eat junk foods these days cuz I start focusing more on my health.")

{'not_food': 0.9973516464233398, 'food': 0.0026483156252652407}

# Building a small Gradio demo to run locally

In [4]:
import gradio as gr

# 1. Setup a gradio interface to accept text and output labels
demo = gr.Interface(
    fn=food_not_food_classifier,
    inputs='text',
    outputs=gr.Label(num_top_classes=2), # show top 2 classes
    title="Food or Not Food Classifier ",
    description="A text classifier to determine if a sentence is about food or not food.",
    examples=[["I whipped up a fresh batch of code, but it seems to have a syntax error.",
               "A delicious photo of a plate of scrambled eggs, bacon and toast."]]
)

# 2. Launch the interface
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d108e1bbbd0f4b9d8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Making a demo publicly accessible

In [7]:
from pathlib import Path

# Make a directory for demos
demos_dir = Path("../demos")
demos_dir.mkdir(exist_ok=True)

# Create a folder for food_not_food_text_classifier demo
food_not_food_text_classifier_demo = Path(demos_dir, "food_not_food_text_classifier")
food_not_food_text_classifier_demo.mkdir(exist_ok=True)

## Creating `app.py`

In [ ]:
%%writefile ../demos/food_not_food_text_classifier/app.py

# 1. Import the required packages
import os
import torch
import gradio as gr

from typing import Dict
from transformers import pipeline

# 2. Define configuration variables OUTSIDE the function
HUGGING_FACE_MODEL_PATH = os.getenv(
    "MODEL_PATH",
    "SaiePaine/learn_hf_food_not_food_text_classifier-distilbert-base-uncased"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# 3. Initialize the pipeline ONCE outside the function (at application startup)
# It loads the model into memory exactly once.
classifier_pipeline = pipeline(
    task='text-classification',
    model=HUGGING_FACE_MODEL_PATH,
    device=DEVICE,
    top_k=None,
    batch_size=32
)


# 4. Create a function which takes text as input
def food_not_food_classifier(text: str) -> Dict[str, float]:
  """
  Take an input string of text and classifies it into food/not_food in the forms of a dictionary.
  """

  # Get outputs from pipeline (as a list of dicts)
  outputs = classifier_pipeline(text)[0]

  # Format output for Gradio (e.g. {"label_1": probability_1, "label_2": probability_2})
  output_dict = {}
  for item in outputs:
    output_dict[item['label']] = item["score"]

  return output_dict


# 5. Create a Gradio Interface with details about my app
description = """
A text classifier to determine if a sentence is about food or not food.

Fine-tuned from [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) on a [small dataset of food and not food text](https://huggingface.co/datasets/mrdbourke/learn_hf_food_not_food_image_captions).
"""

demo = gr.Interface(
    fn=food_not_food_classifier,
    inputs='text',
    outputs=gr.Label(num_top_classes=2),
    title="🍗🚫🥑 Food or Not Food Text Classifier",
    description=description,
    examples=[["The chef prepared a perfectly seared ribeye steak served alongside garlic mashed potatoes."],
     ["I had to restart my router because the Wi-Fi connection kept dropping during the call."]])

# 6. Launch the interface
if __name__ == "__main__":
  demo.launch()

## Creating `requirements.txt`

In [9]:
%%writefile ../demos/food_not_food_text_classifier/requirements.txt
gradio
torch
transformers
accelerate

Writing ../demos/food_not_food_text_classifier/requirements.txt


## Creating `README.md`

In [10]:
%%writefile ../demos/food_not_food_text_classifier/README.md
---
title: Food Not Food Text Classifier
emoji: 🍗🚫🥑
colorFrom: blue
colorTo: yellow
sdk: gradio
app_file: app.py
pinned: false
license: apache-2.0
---

# 🍗🚫🥑 Food Not Food Text Classifier

Small demo to showcase a text classifier to determine if a sentence is about food or not food.

DistillBERT model fine-tuned on a small synthetic dataset of 250 generated [Food or Not Food image captions](https://huggingface.co/datasets/mrdbourke/learn_hf_food_not_food_image_captions).


Writing ../demos/food_not_food_text_classifier/README.md


In [11]:
!ls ../demos/food_not_food_text_classifier

app.py	README.md  requirements.txt


## Uploading the demo to Hugging Face Spaces

In [12]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:
# 1. Import the required methods for uploading to the Hugging Face Hub
from huggingface_hub import(
    create_repo,
    get_full_repo_name,
    upload_file,
    upload_folder
)

# 2. Defind the parameters that are used for the upload
LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD = "../demos/food_not_food_text_classifier"
HF_TARGET_SPACE_NAME = "learn_hf_food_not_food_text_classifier_demo"
HF_REPO_TYPE = "space"
HF_SPACE_SDK = "gradio"
HF_TOKEN = ""

# 3. Create a space repository on the Hugging Face Hub
print(f"[INFO] Creating repo on Hugging Face Hub with name: {HF_TARGET_SPACE_NAME}")
create_repo(
    repo_id = HF_TARGET_SPACE_NAME,
    repo_type = HF_REPO_TYPE,
    private=False,
    space_sdk=HF_SPACE_SDK, # Changed 'space' to 'space_sdk'
    exist_ok=True
)

# 4. Get the full repository name (e.g. )
full_hf_repo_name = get_full_repo_name(model_id=HF_TARGET_SPACE_NAME)
print(f"[INFO] Full Hugging Face Hub repo name: {full_hf_repo_name}")

# 5. Upload our demo folder
print(f"[INFO] Uploading {LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD} to repo: {full_hf_repo_name}")
folder_upload_url = upload_folder(
    repo_id = full_hf_repo_name,
    folder_path = LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD,
    path_in_repo = '.',
    repo_type = HF_REPO_TYPE,
    commit_message = "Uploading food not food text classifier demo app.py"
)

print(f"[INFO] Demo folder successfully uploaded with commit URL: {folder_upload_url}")

[INFO] Creating repo on Hugging Face Hub with name: learn_hf_food_not_food_text_classifier_demo
[INFO] Full Hugging Face Hub repo name: SaiePaine/learn_hf_food_not_food_text_classifier_demo
[INFO] Uploading ../demos/food_not_food_text_classifier to repo: SaiePaine/learn_hf_food_not_food_text_classifier_demo
[INFO] Demo folder successfully uploaded with commit URL: https://huggingface.co/spaces/SaiePaine/learn_hf_food_not_food_text_classifier_demo/commit/ac0b5f2d051437cdcf89e2074af8d0246385f821
